
# DATA 304 — Demo: Advanced Text Representations

This notebook demonstrates:
- Latent Semantic Analysis (LSA)
- Latent Dirichlet Allocation (LDA)
- Word embeddings (Word2Vec)
- Contextual sentence embeddings (BERT via Sentence Transformers)
- Applying different representations in a simple sentiment pipeline


## 1. Setup

In [16]:
# !pip install gensim sentence_transformers

In [1]:

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
# %matplotlib inline


## 2. Toy Corpus for Topics / Concepts

In [2]:

docs = [
    "Data wrangling with pandas and Python for messy datasets",
    "Cleaning text data and removing noise is essential",
    "Machine learning models use numeric features like TF IDF",
    "Topic models like LDA discover themes in document collections",
    "LSA reduces TF IDF into latent concepts for similarity search",
    "Word embeddings capture semantic similarity between words",
    "BERT provides contextual sentence embeddings for NLP tasks",
    "Clustering documents by topic helps organize large corpora"
]

len(docs), docs[:3]


(8,
 ['Data wrangling with pandas and Python for messy datasets',
  'Cleaning text data and removing noise is essential',
  'Machine learning models use numeric features like TF IDF'])

## 3. TF-IDF Representation (Baseline)

In [3]:

tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out()).round(3)
tfidf_df.head()


,bert,capture,cleaning,clustering,collections,concepts,contextual,corpora,data,datasets,...,similarity,tasks,text,tf,themes,topic,use,word,words,wrangling
0,0.0,0.0,0.000,0.0,0.000,0.000,0.0,0.0,0.351,0.419,...,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.419
1,0.0,0.0,0.419,0.0,0.000,0.000,0.0,0.0,0.351,0.000,...,0.000,0.0,0.419,0.000,0.000,0.000,0.000,0.0,0.0,0.000
2,0.0,0.0,0.000,0.0,0.000,0.000,0.0,0.0,0.000,0.000,...,0.000,0.0,0.000,0.300,0.000,0.000,0.358,0.0,0.0,0.000
3,0.0,0.0,0.000,0.0,0.375,0.000,0.0,0.0,0.000,0.000,...,0.000,0.0,0.000,0.000,0.375,0.314,0.000,0.0,0.0,0.000
4,0.0,0.0,0.000,0.0,0.000,0.375,0.0,0.0,0.000,0.000,...,0.314,0.0,0.000,0.314,0.000,0.000,0.000,0.0,0.0,0.000


## 4. Latent Semantic Analysis (LSA)

In [4]:
# Reduce TF-IDF to a small number of latent concepts
svd = TruncatedSVD(n_components=2, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

lsa_df = pd.DataFrame(X_lsa, columns=["Concept1", "Concept2"])
lsa_df

,Concept1,Concept2
0,-6.169502e-17,5.004031e-17
1,-1.068700e-15,-1.311432e-15
2,7.314129e-01,-1.698727e-01
3,5.426151e-01,-4.344074e-01
4,5.818679e-01,3.036794e-01
5,2.714653e-01,6.714737e-01
6,1.092825e-01,5.400204e-01
7,1.904885e-01,-3.046627e-01


In [5]:
# Reduce TF-IDF to a small number of latent concepts
svd = TruncatedSVD(n_components=3, random_state=42)
X_lsa = svd.fit_transform(X_tfidf)

lsa_df = pd.DataFrame(X_lsa, columns=["Concept1", "Concept2", "Concept3"])
lsa_df

,Concept1,Concept2,Concept3
0,2.921055e-16,-5.796905e-17,7.493905e-01
1,-5.465101e-16,2.943271e-16,7.493905e-01
2,7.314129e-01,-1.698727e-01,1.347709e-16
3,5.426151e-01,-4.344074e-01,6.648746e-16
4,5.818679e-01,3.036794e-01,-2.547394e-16
5,2.714653e-01,6.714737e-01,-2.346334e-16
6,1.092825e-01,5.400204e-01,1.246452e-15
7,1.904885e-01,-3.046627e-01,1.018983e-15


In [6]:
# Inspect top terms for each latent concept
terms = tfidf.get_feature_names_out()
n_top = 8

for i, comp in enumerate(svd.components_):
    top_ids = comp.argsort()[-n_top:][::-1]
    top_terms = [terms[j] for j in top_ids]
    print(f"Concept {i+1}: {', '.join(top_terms)}")


Concept 1: tf, idf, models, like, similarity, use, numeric, learning
Concept 2: embeddings, similarity, semantic, word, words, capture, sentence, tasks
Concept 3: data, removing, essential, noise, text, cleaning, wrangling, python


## 5. Latent Dirichlet Allocation (LDA)

In [7]:
# LDA works on count (BoW) matrix, not TF-IDF
vec = CountVectorizer(max_df=0.95, min_df=1, stop_words='english')
X_counts = vec.fit_transform(docs)

lda = LatentDirichletAllocation(n_components=4, random_state=42)
lda.fit(X_counts)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",4
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [8]:
# View top words per topic
terms = vec.get_feature_names_out()
n_top = 8

for i, comp in enumerate(lda.components_):
    top_ids = comp.argsort()[-n_top:][::-1]
    top_terms = [terms[j] for j in top_ids]
    print(f"Topic {i+1}: {', '.join(top_terms)}")

Topic 1: topic, like, models, themes, lda, discover, document, collections
Topic 2: embeddings, provides, sentence, nlp, tasks, bert, contextual, topic
Topic 3: data, python, datasets, pandas, wrangling, messy, embeddings, topic
Topic 4: tf, idf, similarity, numeric, use, machine, learning, features


In [9]:
# Document–topic distributions
doc_topic = lda.transform(X_counts)
pd.DataFrame(doc_topic.round(3),
             columns=[f"Topic{i+1}" for i in range(lda.n_components)])

,Topic1,Topic2,Topic3,Topic4
0,0.036,0.036,0.893,0.036
1,0.892,0.036,0.036,0.036
2,0.026,0.025,0.025,0.924
3,0.916,0.028,0.028,0.028
4,0.028,0.028,0.028,0.917
5,0.036,0.037,0.036,0.892
6,0.031,0.906,0.031,0.031
7,0.032,0.031,0.031,0.906


## 6. Word Embeddings (Word2Vec)

In [10]:
# Simple tokenization for demo purposes
tokenized_texts = [doc.lower().split() for doc in docs]
tokenized_texts[:2]

[['data',
  'wrangling',
  'with',
  'pandas',
  'and',
  'python',
  'for',
  'messy',
  'datasets'],
 ['cleaning', 'text', 'data', 'and', 'removing', 'noise', 'is', 'essential']]

In [15]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    sg=1  # use skip-gram
)

w2v_model.wv.most_similar("embeddings")

[('word', 0.18911118805408478),
 ('bert', 0.18845707178115845),
 ('datasets', 0.18365265429019928),
 ('by', 0.1610146462917328),
 ('nlp', 0.1593044251203537),
 ('helps', 0.1375955194234848),
 ('words', 0.12816648185253143),
 ('documents', 0.12276550382375717),
 ('features', 0.11792172491550446),
 ('discover', 0.094502754509449)]

In [16]:
# Inspect similarity between some technical terms if present
for pair in [("data", "pandas"),
             ("lda", "topic"),
             ("bert", "embeddings")]:
    w1, w2 = pair
    if w1 in w2v_model.wv.key_to_index and w2 in w2v_model.wv.key_to_index:
        print(f"cosine_sim({w1}, {w2}) = {w2v_model.wv.similarity(w1, w2):.3f}")

cosine_sim(data, pandas) = -0.015
cosine_sim(lda, topic) = -0.050
cosine_sim(bert, embeddings) = 0.188


## 7. Contextual Sentence Embeddings (BERT via Sentence Transformers)

In [17]:
# This cell requires the sentence-transformers package to be installed.
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "He sat by the bank of the river.",
    "He deposited money in the bank.",
    "Data wrangling with pandas is powerful."
]

embeddings = st_model.encode(sentences)
embeddings.shape

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 41.01it/s]


(3, 384)

In [18]:

# Compare similarity between different uses of 'bank'
sim_matrix = cosine_similarity(embeddings)
pd.DataFrame(sim_matrix.round(3),
             index=[f"Sent{i+1}" for i in range(len(sentences))],
             columns=[f"Sent{i+1}" for i in range(len(sentences))])


,Sent1,Sent2,Sent3
Sent1,1.000,0.580,-0.075
Sent2,0.580,1.000,-0.054
Sent3,-0.075,-0.054,1.000


## 8. Simple Sentiment Pipeline with Different Representations

### 8.1 Toy sentiment dataset

In [19]:
import nltk
nltk.download("movie_reviews")

from nltk.corpus import movie_reviews
import numpy as np

texts = [" ".join(movie_reviews.words(fileid)) 
         for fileid in movie_reviews.fileids()]

y = np.array([1 if fileid.startswith("pos") else 0 
              for fileid in movie_reviews.fileids()])


[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [20]:
texts[:5]

['plot : two teen couples go to a church party , drink and then drive . they get into an accident . one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . what \' s the deal ? watch the movie and " sorta " find out . . . critique : a mind - fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn \' t snag this one correctly . they seem to have taken this pretty neat concept , but executed it terribly . so what are the problems with the movie ? well , its main problem is that it \' s simply too jumbled . it starts off " normal " but then downshifts into this " fantasy " world in which you , as an audience member , have no

In [32]:
y[:5]

array([0, 0, 0, 0, 0])

### 8.2 TF-IDF + Logistic Regression (baseline)

In [21]:

tfidf_sent = TfidfVectorizer(stop_words='english')
X_tfidf_sent = tfidf_sent.fit_transform(texts)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf_sent, y, test_size=0.25, random_state=42
)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train, y_train)

print("TF-IDF accuracy:", clf_tfidf.score(X_test, y_test))


TF-IDF accuracy: 0.802


### 8.3 LSA (reduced concepts) + Logistic Regression

In [34]:

svd_sent = TruncatedSVD(n_components=2, random_state=42)
X_lsa_sent = svd_sent.fit_transform(X_tfidf_sent)

X_train_lsa, X_test_lsa, y_train_lsa, y_test_lsa = train_test_split(
    X_lsa_sent, y, test_size=0.25, random_state=42
)

clf_lsa = LogisticRegression(max_iter=1000)
clf_lsa.fit(X_train_lsa, y_train_lsa)

print("LSA accuracy:", clf_lsa.score(X_test_lsa, y_test_lsa))


LSA accuracy: 0.576


### 8.4 Word2Vec document embeddings + Logistic Regression

In [35]:

# Train a small Word2Vec on sentiment texts
sent_tokenized = [t.lower().split() for t in texts]

w2v_sent_model = Word2Vec(
    sentences=sent_tokenized,
    vector_size=50,
    window=5,
    min_count=1,
    workers=2,
    sg=1
)

def doc_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv.key_to_index]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X_w2v_sent = np.vstack([doc_vector(tokens, w2v_sent_model)
                        for tokens in sent_tokenized])

X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v_sent, y, test_size=0.25, random_state=42
)

clf_w2v = LogisticRegression(max_iter=1000)
clf_w2v.fit(X_train_w2v, y_train_w2v)

print("Word2Vec accuracy:", clf_w2v.score(X_test_w2v, y_test_w2v))


Word2Vec accuracy: 0.648


### 8.5 BERT embeddings

In [36]:
from sentence_transformers import SentenceTransformer

# 1) Build sentence embeddings with a small, fast BERT model
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

X_bert = bert_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 2) Train/test split (same pattern as before)
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_bert,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3) Train Logistic Regression on BERT embeddings
clf_bert = LogisticRegression(max_iter=2000)
clf_bert.fit(X_train_bert, y_train_bert)

print("BERT accuracy:", clf_bert.score(X_test_bert, y_test_bert))

Batches: 100%|██████████| 63/63 [03:28<00:00,  3.30s/it]

BERT accuracy: 0.6775
